<a href="https://colab.research.google.com/github/MengOonLee/Deep_learning/blob/master/TensorFlow/Probabilistic/Distributions/Broadcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Broadcasting rules

This reading will introduce you to numpy's broadcasting rules and show how you can use broadcasting when specifying batches of distributions in TensorFlow, as well as with the `prob` and `log_prob` methods.

Broadcasting will also be discussed and demonstrated in the following videos.

In [ ]:
%%bash
pip install --no-cache-dir -qU pip
pip install --no-cache-dir -qU \
    numpy pandas matplotlib seaborn \
    scipy scikit-learn \
    torch torchvision lightning
pip check

## Operations on arrays of different sizes in numpy

Numpy operations can be applied to arrays that are not of the same shape, but only if the shapes satisfy certain conditions.

As a demonstration of this, let us add together two arrays of different shapes:

In [1]:
# Add two arrays with different shapes
import numpy as np
import torch

np.random.seed(seed=42)
torch.manual_seed(seed=42)

a = np.array([[1], [2], [3], [4]], dtype=np.float32) # shape (4, 1)
a = torch.from_numpy(a)

b = np.array([0, 1, 2], dtype=np.float32)  # shape (3,)
b = torch.from_numpy(b)

a + b

tensor([[1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 5.],
        [4., 5., 6.]])

This is the addition

    [ [1.],    +  [0., 1., 2.]  
      [2.],  
      [3.],  
      [4.] ]

To execute it, numpy:
1. Aligned the shapes of `a` and `b` on the last axis and prepended 1s to the shape with fewer axes:
        a: 4 x 1     --->    a: 4 x 1
        b:     3     --->    b: 1 x 3
        

2. Checked that the sizes of the axes matched or were equal to 1:
        a: 4 x 1  
        b: 1 x 3
`a` and `b` satisfied this criterion.


3. Stretched both arrays on their 1-valued axes so that their shapes matched, then added them together.  
`a` was replicated 3 times in the second axis, while `b` was replicated 4 times in the first axis.

This meant that the addition in the final step was

    [ [1., 1., 1.],    +  [ [0., 1., 2.],  
      [2., 2., 2.],         [0., 1., 2.],  
      [3., 3., 3.],         [0., 1., 2.],  
      [4., 4., 4.] ]        [0., 1., 2.] ]
      
Addition was then carried out element-by-element, as you can verify by referring back to the output of the code cell above.  
This resulted in an output with shape 4 x 3.


## Numpy's broadcasting rule

Broadcasting rules describe how values should be transmitted when the inputs to an operation do not match.  
In numpy, the broadcasting rule is very simple:
> Prepend 1s to the smaller shape,   
check that the axes of both arrays have sizes that are equal or 1,  
then stretch the arrays in their size-1 axes.

A crucial aspect of this rule is that it does not require the input arrays have the same number of axes.  
Another consequence of it is that a broadcasting output will have the largest size of its inputs in each axis.  
Take the following multiplication as an example:

        a: 3 x 7 x 1  
        b:     1 x 5  
    a * b: 3 x 7 x 5

You can see that the output shape is the maximum of the sizes in each axis.

Numpy's broadcasting rule also does not require that one of the arrays has to be bigger in all axes.  
This is seen in the following example, where `a` is smaller than `b` in its third axis but is bigger in its second axis.

In [2]:
# Multiply two arrays with different shapes
import numpy as np
import torch

np.random.seed(seed=42)
torch.manual_seed(seed=42)

a = np.array([[[0.01], [0.1]], [[1], [10]]], dtype=np.float32) # shape (2, 2, 1)
a = torch.from_numpy(a)

b = np.array([[[2, 2]], [[3, 3]]], dtype=np.float32) # shape (2, 1, 2)
b = torch.from_numpy(b)

a * b # shape (2, 2, 2)

tensor([[[2.0000e-02, 2.0000e-02],
         [2.0000e-01, 2.0000e-01]],

        [[3.0000e+00, 3.0000e+00],
         [3.0000e+01, 3.0000e+01]]])

Broadcasting behaviour also points to an efficient way to compute an outer product in numpy:

In [3]:
# Use broadcasting to compute an outer product
import numpy as np
import torch

np.random.seed(seed=42)
torch.manual_seed(seed=42)

a = np.array([-1, 0, 1], dtype=np.float32)
a = torch.from_numpy(a)

b = np.array([0, 1, 2, 3], dtype=np.float32)
b = torch.from_numpy(b)

a[:, None] * b  # outer product ab^T, where a and b are column vectors

tensor([[-0., -1., -2., -3.],
        [ 0.,  0.,  0.,  0.],
        [ 0.,  1.,  2.,  3.]])

The idea of numpy stretching the arrays in their size-1 axes is useful and is functionally correct. But this is not what numpy literally does behind the scenes, since that would be an inefficient use of memory. Instead, numpy carries out the operation by looping over singleton (size-1) dimensions.

To give you some practise with broadcasting, try predicting the output shapes for the following operations:

In [4]:
# Define three arrays with different shapes
import numpy as np
import torch

np.random.seed(seed=42)
torch.manual_seed(seed=42)

a = np.array([[1], [2], [3]], dtype=np.float32)
a = torch.from_numpy(a)

b = np.zeros(shape=(10, 1, 1), dtype=np.float32)
b = torch.from_numpy(b)

c = np.ones(shape=(4), dtype=np.float32)
c = torch.from_numpy(c)

In [5]:
# Predict the shape before executing this cell

(a + b).shape

torch.Size([10, 3, 1])

In [6]:
# Predict the shape before executing this cell

(a * c).shape

torch.Size([3, 4])

In [7]:
# Predict the shape before executing this cell

(a * b + c).shape

torch.Size([10, 3, 4])

## Broadcasting for univariate TensorFlow Distributions

The broadcasting rule for TensorFlow is the same as that for numpy. For example, TensorFlow also allows you to specify the parameters of Distribution objects using broadcasting.

What is meant by this can be understood through an example with the univariate normal distribution. Say that we wish to specify a parameter grid for six Gaussians. The parameter combinations to be used, `(loc, scale)`, are:  

    (0, 1)  
    (0, 10)  
    (0, 100)  
    (1, 1)  
    (1, 10)  
    (1, 100)
    
A laborious way of doing this is to explicitly pass each parameter to `tfd.Normal`:

In [1]:
# Define a batch of Normal distributions without broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[0., 0., 0., 1., 1., 1.])
scale = torch.tensor(data=[1., 10., 100., 1., 10., 100.])
normal= torch.distributions.Normal(loc=loc, scale=scale)

# Print the distribution and notice the batch and event shapes
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([6])
Event shape: torch.Size([])


In [2]:
# Check the parameter values for loc

normal.loc

tensor([0., 0., 0., 1., 1., 1.])

In [3]:
# Check the parameter values for scale

normal.scale

tensor([  1.,  10., 100.,   1.,  10., 100.])

A more succinct way to create a batch of distributions for this parameter grid is to use broadcasting.  
Consider what would happen if we were to broadcast these arrays according the rule discussed earlier:
    
    loc = [ [0.],
            [1.] ]
    scale = [1., 10., 100.]
    
The shapes would be stretched according to

    loc:   2 x 1 ---> 2 x 3
    scale: 1 x 3 ---> 2 x 3
    
resulting in

    loc = [ [0., 0., 0.],
            [1., 1., 1.] ]
    scale = [ [1., 10., 100.],
              [1., 10., 100.] ]
              
which are compatible with the `loc` and `scale` arguments of `tfd.Normal`.  
Sure enough, this is precisely what TensorFlow does:

In [4]:
# Define a batch of Normal distributions with broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[[0.], [1.]])
scale = torch.tensor(data=[1., 10., 100.])

normal = torch.distributions.Normal(loc=loc, scale=scale)

# Print the distribution and notice the batch and event shapes
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([2, 3])
Event shape: torch.Size([])


In [2]:
# The stored loc parameter values are what you pass in, not what is used after broadcasting

normal.loc

tensor([[0., 0., 0.],
        [1., 1., 1.]])

In [3]:
# The stored scale parameter values are what you pass in, not what is used after broadcasting

normal.scale

tensor([[  1.,  10., 100.],
        [  1.,  10., 100.]])

In summary, TensorFlow broadcasts parameter arrays: it stretches them according to the broadcasting rule, then creates a distribution on an element-by-element basis.

#### Broadcasting with `prob` and `log_prob` methods

When using `prob` and  `log_prob` with broadcasting, we follow the same principles as before. Let's make a new batch of normals as before but with means which are centered at different locations to help distinguish the results we get.

In [5]:
# Define a batch of Normal distributions with broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[[0.], [10.]])
scale = torch.tensor(data=[1., 1., 1.])

normal = torch.distributions.Normal(loc=loc, scale=scale)
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([2, 3])
Event shape: torch.Size([])


We can feed in samples of any shape as long as it can be broadcast agasint our batch shape for this example.

In [2]:
# Use broadcasting along the second axis with the prob method
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(2, 1), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([[2.7032e-01, 2.7032e-01, 2.7032e-01],
        [4.7664e-19, 4.7664e-19, 4.7664e-19]])

Or boradcasting along the first axis instead:

In [3]:
# Use broadcasting along the first axis with the prob method
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(1, 3), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([[2.7032e-01, 2.6249e-01, 3.7075e-01],
        [3.5384e-19, 4.7664e-19, 3.2894e-21]])

Or even both axes:

In [4]:
# Use broadcasting along both axes with the prob method
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(1, 1), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([[2.7032e-01, 2.7032e-01, 2.7032e-01],
        [3.5384e-19, 3.5384e-19, 3.5384e-19]])

`log_prob` works in the exact same way with broadcasting. We can replace `prob` with `log_prob` in any of the previous examples:

In [5]:
# Use broadcasting along the first axis with the log_prob method
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(1, 3), dtype=torch.float32)
normal.log_prob(value=value)

tensor([[ -1.3081,  -1.3376,  -0.9922],
        [-42.4854, -42.1875, -47.1636]])

## Broadcasting for multivariate TensorFlow distributions

Broadcasting behaviour for multivariate distributions is only a little more sophisticated than it is for univariate distributions.

Recall that `MultivariateNormalDiag` has two parameter arguments: `loc` and `scale_diag`. When specifying a single distribution, these arguments are vectors of the same length:

In [1]:
# Define a multivariate Gaussian distribution without broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[0., 0.])
scale_diag = torch.tensor(data=[1., 0.5])
sigma = torch.diag_embed(input=scale_diag**2)
scale_tril = torch.linalg.cholesky(input=sigma)
normal = torch.distributions.MultivariateNormal(loc=loc, scale_tril=scale_tril)

print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([])
Event shape: torch.Size([2])


In [2]:
# Print the loc parameter

normal.loc

tensor([0., 0.])

In [3]:
# Print the covariance matrix - the diagonal is scale_diag^2

normal.covariance_matrix

tensor([[1.0000, 0.0000],
        [0.0000, 0.2500]])

The size of the final axis of the inputs determines the event shape for each distribution in the batch.  This means that if we pass
    
    loc = [ [0., 0.],
            [1., 1.] ]
    scale_diag = [1., 0.5]
    
such that

    loc:        2 x 2
    scale_diag: 1 x 2
                    ^ final dimension is interpreted as event dimension
                ^ other dimensions are interpreted as batch dimensions  
then a batch of two bivariate normal distributions will be created.

In [2]:
# Define a multivariate Gaussian distribution with broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[[0.], [1.]])
loc = loc.expand(-1, 2)
scale_diag = torch.tensor(data=[1., 0.5])
sigma = torch.diag_embed(input=scale_diag**2)
scale_tril = torch.linalg.cholesky(input=sigma)

normal = torch.distributions.MultivariateNormal(loc=loc, scale_tril=scale_tril)

# Print the distribution - note the event_shape and batch_shape
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([2])
Event shape: torch.Size([2])


In [2]:
# Print the distribution parameters
# There is a batch of two distributions with different means and same covariance

print("Means:\n", normal.loc)
print("\nCovariance matrices:\n", normal.covariance_matrix)

Means:
 tensor([[0., 0.],
        [1., 1.]])

Covariance matrices:
 tensor([[[1.0000, 0.0000],
         [0.0000, 0.2500]],

        [[1.0000, 0.0000],
         [0.0000, 0.2500]]])


Knowing that, for multivariate distributions, TensorFlow

- interprets the final axis of an array of parameters as the event shape,


- and broadcasts over the remaining axes,  

can you predict what the batch and event shapes will if we pass the arguments


    loc = [ [ 1.,  1.,  1.],
            [-1., -1., -1.] ] # shape (2, 3)
    scale_diag = [ [[0.1, 0.1, 0.1]],
                   [[10., 10., 10.]] ] # shape (2, 1, 3)
                   
to `MultivariateNormalDiag`?

Solution:

Align the parameter array shapes on their last axis, prepending 1s where necessary:  
    
           loc: 1 x 2 x 3  
    scale_diag: 2 x 1 x 3  

The final axis has size 3, so `event_shape = (3)`. The remaining axes are broadcast over to yield  
    
           loc: 2 x 2 x 3  
    scale_diag: 2 x 2 x 3  

so `batch_shape = (2, 2)`.

Let's see if this is correct!

In [3]:
# Define a multivariate Gaussian distribution with broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[[1.], [-1.]])
loc = loc.expand(-1, 3) # shape (2, 3)

scale_diag = torch.tensor(data=[[[0.1]], [[10.]]])
scale_diag = scale_diag.expand(-1, -1, 3) # shape (2, 1, 3)
sigma = torch.diag_embed(input=scale_diag**2)
scale_tril = torch.linalg.cholesky(input=sigma)

normal = torch.distributions.MultivariateNormal(loc=loc, scale_tril=scale_tril)

# Print the distribution and note batch and event shapes - bingo!
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([2, 2])
Event shape: torch.Size([3])


In [2]:
# Print the distribution parameters

print("Means:\n", normal.loc)
print("\nCovariance matrices:\n", normal.covariance_matrix)

Means:
 tensor([[[ 1.,  1.,  1.],
         [-1., -1., -1.]],

        [[ 1.,  1.,  1.],
         [-1., -1., -1.]]])

Covariance matrices:
 tensor([[[[1.0000e-02, 0.0000e+00, 0.0000e+00],
          [0.0000e+00, 1.0000e-02, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 1.0000e-02]],

         [[1.0000e-02, 0.0000e+00, 0.0000e+00],
          [0.0000e+00, 1.0000e-02, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 1.0000e-02]]],


        [[[1.0000e+02, 0.0000e+00, 0.0000e+00],
          [0.0000e+00, 1.0000e+02, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 1.0000e+02]],

         [[1.0000e+02, 0.0000e+00, 0.0000e+00],
          [0.0000e+00, 1.0000e+02, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 1.0000e+02]]]])


As we did before lets also look at broadcasting when we have batches of multivariate distributions.

In [1]:
# Define a batch of Normal distributions with broadcasting
import torch

torch.manual_seed(seed=42)

loc = torch.tensor(data=[[0.], [1.], [0.]])
scale = torch.tensor(data=[1., 10., 100., 1., 10., 100.])

normal = torch.distributions.Normal(loc=loc, scale=scale)
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([3, 6])
Event shape: torch.Size([])


And to refresh our memory of `Independent` we'll use it below to roll the rightmost batch shape into the event shape.

In [2]:
# Create a multivariate Independent distribution
import torch

torch.manual_seed(seed=42)

normal = torch.distributions.Independent(base_distribution=normal,
    reinterpreted_batch_ndims=1)
print("Batch shape:", normal.batch_shape)
print("Event shape:", normal.event_shape)

Batch shape: torch.Size([3])
Event shape: torch.Size([6])


Now, onto the broadcasting:

In [3]:
# Use broadcasting with the prob method
# B shaped input (broadcast over event)
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(3, 1), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([1.8365e-09, 4.0021e-09, 3.4766e-09])

In [4]:
# Use broadcasting with the prob method
# E shaped input (broadcast over batch)
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(1, 6), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([1.7157e-09, 3.9926e-09, 1.7157e-09])

In [5]:
# Use broadcasting with the prob method
# [S,B,E] shaped input (broadcast over samples)
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(2, 3, 6), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([[1.7157e-09, 2.0998e-09, 2.5053e-09],
        [3.5210e-09, 1.9535e-09, 2.1612e-09]])

In [6]:
# [S,b,e] shaped input where [b,e] can be broadcast agaisnt [B,E]
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(2, 1, 6), dtype=torch.float32)
torch.exp(normal.log_prob(value=value))

tensor([[1.7157e-09, 3.9926e-09, 1.7157e-09],
        [3.8373e-09, 2.0998e-09, 3.8373e-09]])

As a final example with `log_prob` instead of `prob`

In [7]:
# Use broadcasting with the log_prob method
# [S,b,e] shaped input where [b,e] can be broadcast agaisnt [B,E]
import torch

torch.manual_seed(seed=42)

value = torch.rand(size=(2, 3, 1), dtype=torch.float32)
normal.log_prob(value=value)

tensor([[-20.1154, -19.3364, -19.4772],
        [-20.2587, -19.7044, -19.6939]])

You should now feel confident specifying batches of distributions using broadcasting. As you may have already guessed, broadcasting is especially useful when specifying grids of hyperparameters.

If you don't feel entirely comfortable with broadcasting quite yet, don't worry: re-read this notebook, go through the further reading provided below, and experiment with broadcasting in both numpy and TensorFlow, and you'll be broadcasting in no time.

### Further reading and resources
* Numpy documentation on broadcasting: https://numpy.org/devdocs/user/theory.broadcasting.html
* https://www.tensorflow.org/xla/broadcasting